# 🧹 BlueStock Mutual Fund Analytics — 02. Data Cleaning & Transformation
**Capstone Deliverable D1/D3 Prerequisite**  
**Author:** BlueStock Quantitative Analytics Team  
**Scope:** ISO Date Standardization | AMFI Normalization | Full Daily NAV Calendar Reindexing & Forward-Fill | Anomaly Handling

---

### Cleaning & Transformation Objectives
1. **Path Agnostic Setup:** Use `pathlib.Path` to eliminate hardcoded absolute paths.
2. **Date Standardization:** Convert heterogeneous date strings to ISO `YYYY-MM-DD`.
3. **AMFI Code Normalization:** Strip whitespace and resolve float conversion anomalies (`.0`).
4. **Weekend & Holiday Handling:** Build a full daily continuous calendar per scheme and apply forward-fill (`ffill()`) to eliminate weekend gaps.
5. **Investor Transaction Cleaning:** Harmonize categorical fields (KYC, payment mode, city tier) and validate amounts.
6. **Persistence:** Write cleaned and validated CSV datasets into `data/processed/`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def normalize_amfi(series):
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

def to_iso_date(series):
    return pd.to_datetime(series, errors="coerce").dt.strftime("%Y-%m-%d")

print("Setup completed successfully.")


Setup completed successfully.


### 1. Clean Fund Master (`clean_fund_master.csv`)

In [2]:
df_funds = pd.read_csv(RAW_DIR / "01_fund_master.csv")
df_funds["amfi_code"] = normalize_amfi(df_funds["amfi_code"])
df_funds["launch_date"] = to_iso_date(df_funds["launch_date"])
df_funds["expense_ratio_pct"] = pd.to_numeric(df_funds["expense_ratio_pct"], errors="coerce").fillna(1.25)
df_funds["exit_load_pct"] = pd.to_numeric(df_funds["exit_load_pct"], errors="coerce").fillna(1.0)
df_funds["min_sip_amount"] = pd.to_numeric(df_funds["min_sip_amount"], errors="coerce").fillna(500.0)
df_funds["min_lumpsum_amount"] = pd.to_numeric(df_funds["min_lumpsum_amount"], errors="coerce").fillna(1000.0)
df_funds.drop_duplicates(subset=["amfi_code"], inplace=True)

out_funds = PROCESSED_DIR / "clean_fund_master.csv"
df_funds.to_csv(out_funds, index=False)
print(f"Saved {len(df_funds)} cleaned fund records to {out_funds.name}")
display(df_funds.head(3))


Saved 40 cleaned fund records to clean_fund_master.csv


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03


### 2. Clean Historical NAVs & Reindex Full Calendar (ffill)
> **Crucial Rule:** Avoid skipping weekends and market holidays. Always reindex to a complete daily date range and forward-fill (`ffill()`) so that rolling metrics, holding periods, and CAGR calculations are mathematically consistent.


In [3]:
df_nav = pd.read_csv(RAW_DIR / "02_nav_history.csv")
df_nav["amfi_code"] = normalize_amfi(df_nav["amfi_code"])
df_nav["date"] = to_iso_date(df_nav["date"])
df_nav["nav"] = pd.to_numeric(df_nav["nav"], errors="coerce")

# Drop nulls, non-positive NAVs, and duplicates
df_nav = df_nav.dropna(subset=["amfi_code", "date", "nav"])
df_nav = df_nav[df_nav["nav"] > 0]
df_nav = df_nav.drop_duplicates(subset=["amfi_code", "date"], keep="last")
df_nav = df_nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# Save standard cleaned NAV history
df_nav.to_csv(PROCESSED_DIR / "clean_nav_history.csv", index=False)
print(f"Cleaned NAV observations: {len(df_nav):,}")

# Generate Full Daily Calendar with Forward-Fill
daily_frames = []
nav_subset = df_nav[["amfi_code", "date", "nav"]].copy()
nav_subset["dt"] = pd.to_datetime(nav_subset["date"])

for amfi_code, grp in nav_subset.groupby("amfi_code"):
    grp = grp.sort_values("dt").set_index("dt")
    full_calendar = pd.date_range(start=grp.index.min(), end=grp.index.max(), freq="D")
    actual_dates = full_calendar.isin(grp.index)
    
    expanded = grp[["nav"]].reindex(full_calendar)
    expanded["nav"] = expanded["nav"].ffill()
    expanded["amfi_code"] = amfi_code
    expanded["is_actual_nav"] = actual_dates
    expanded.index.name = "date"
    daily_frames.append(expanded.reset_index())

daily_nav_calendar = pd.concat(daily_frames, ignore_index=True)
daily_nav_calendar["date"] = daily_nav_calendar["date"].dt.strftime("%Y-%m-%d")
daily_nav_calendar.to_csv(PROCESSED_DIR / "daily_nav_calendar.csv", index=False)

print(f"Expanded Full Daily NAV Calendar: {len(daily_nav_calendar):,} records (forward-filled weekends/holidays)")
display(daily_nav_calendar.head(5))


Cleaned NAV observations: 46,000


Expanded Full Daily NAV Calendar: 64,320 records (forward-filled weekends/holidays)


,date,nav,amfi_code,is_actual_nav
0,2022-01-03,520.4608,100016,True
1,2022-01-04,515.0971,100016,True
2,2022-01-05,521.7239,100016,True
3,2022-01-06,515.7880,100016,True
4,2022-01-07,515.1639,100016,True


### 3. Clean Investor Transactions (`clean_investor_transactions.csv`)

In [4]:
df_tx = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
df_tx["amfi_code"] = normalize_amfi(df_tx["amfi_code"])
df_tx["transaction_date"] = to_iso_date(df_tx["transaction_date"])
df_tx["amount_inr"] = pd.to_numeric(df_tx["amount_inr"], errors="coerce")

df_tx = df_tx.dropna(subset=["investor_id", "amfi_code", "transaction_date", "amount_inr"])
df_tx = df_tx[df_tx["amount_inr"] > 0]
df_tx.drop_duplicates(inplace=True)

df_tx.to_csv(PROCESSED_DIR / "clean_investor_transactions.csv", index=False)
print(f"Cleaned Investor Transactions: {len(df_tx):,} records")
display(df_tx.head(3))


Cleaned Investor Transactions: 32,778 records


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified


### 4. Clean AUM by Fund House (`clean_aum_by_fund_house.csv`)
> **Unit Disambiguation:** Industry AUM is reported in Lakh Crore (₹ Lakh Cr), whereas scheme-level AUM is reported in Crore (₹ Cr). We explicitly create `aum_crore = aum_lakh_crore * 100,000` to prevent order-of-magnitude errors.


In [5]:
df_aum = pd.read_csv(RAW_DIR / "03_aum_by_fund_house.csv")
df_aum["date"] = to_iso_date(df_aum["date"])
df_aum["aum_lakh_crore"] = pd.to_numeric(df_aum["aum_lakh_crore"], errors="coerce")
df_aum["aum_crore"] = df_aum["aum_lakh_crore"] * 100000.0
df_aum["num_schemes"] = pd.to_numeric(df_aum["num_schemes"], errors="coerce").fillna(0).astype(int)

df_aum.to_csv(PROCESSED_DIR / "clean_aum_by_fund_house.csv", index=False)
print(f"Cleaned Fund House AUM: {len(df_aum)} records")
display(df_aum.head(5))


Cleaned Fund House AUM: 90 records


,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000.0,186
1,2022-03-31,ICICI Prudential MF,4.65,465000.0,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000.0,195
3,2022-03-31,Nippon India MF,2.70,270000.0,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000.0,168


### 5. Cleaning Verification Summary
- **No Missing Values in Keys:** AMFI codes and dates are 100% complete and ISO standardized.
- **Continuity Guaranteed:** NAV gaps on non-trading days resolved via `daily_nav_calendar.csv` forward-fill.
- **Unit Precision:** Industry Lakh Crore converted to Crores for uniform relational joins.
- **Datasets Ready:** All 10 cleaned tables are loaded into `data/processed/` and verified for SQLite ingestion.
